In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import pymc as pm
import aesara.tensor as at
from libpysal.weights import KNN
import arviz as az

In [ ]:
# 1. 데이터 불러오기 및 GeoDataFrame 생성
df = pd.read_csv("../../data/final/8_Apis_cerana_with_nearby_species.csv")
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]))
gdf.crs = "EPSG:4326"
gdf = gdf.to_crs("EPSG:3857")

# 2. 변수 설정 (변수명은 실제 컬럼에 맞게 조정)
y = gdf['species_count'].values  # 예: 종수
X = gdf[['elevation', 'temp_mean']].values  # 예: 설명변수

In [ ]:
# 3. 공간 가중치 행렬 (예: K-최근접 이웃 기반)
knn = KNN.from_dataframe(gdf, k=4)
W = knn.full()[0].astype(int)
num_areas = W.shape[0]
num_neighbors = W.sum(axis=1)

# 4. Precision matrix (Q)
D = np.diag(num_neighbors)
Q = D - W
Q += np.eye(num_areas) * 1e-6  # 수치 안정화용

# 5. CAR 모델 정의 및 샘플링
with pm.Model() as car_model:
    beta = pm.Normal("beta", mu=0, sigma=10, shape=X.shape[1])
    tau = pm.Gamma("tau", 2, 2)
    phi = pm.MvNormal("phi", mu=np.zeros(num_areas), tau=tau * Q, shape=num_areas)
    mu = at.dot(X, beta) + phi
    sigma = pm.HalfNormal("sigma", sigma=5)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)
    
    trace = pm.sample(1000, tune=1000, chains=2, return_inferencedata=True)

# 6. 결과 확인
az.summary(trace, var_names=["beta", "tau", "sigma"])